In [ ]:
# Carico la cartella one drive in cui ci sono:
# cartella video nutrition estimation / video annotation nutrition estimation
# cartella video nutrition change / video annotation nutrition change
# cartella video image nutrition estimation/ video annotation image estimation

# dei video scaricati ho fatto un samplig a N=128 FRAME
# caricato su one drive ed adesso linko da questo notebook
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Verifico la scheda video**
- Colab con GPU **L4** (24 GB di VRAM, circa 22 utilizzabili)
- usare una sessione **nuova**: `transformers==4.37.2` e' incompatibile con VideoLLaMA 2 (4.40.0)


In [ ]:
# Runtime: Runtime > Cambia tipo di runtime > GPU L4, sessione NUOVA
!nvidia-smi
import os, sys, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # meno frammentazione della VRAM
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"                    # evita il timeout del secret HF_TOKEN di Colab
os.environ.pop("HF_TOKEN", None)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Nessuna GPU: Runtime > Cambia tipo di runtime > L4"
print("GPU:", gpu, "| Python:", sys.version.split()[0])
if "L4" not in gpu:
    print("ATTENZIONE: il notebook e' tarato su L4. Un'altra GPU va bene se ha almeno 24 GB.")


In [ ]:
import os, sys

# transformers 4.37.2: l'ultima compatibile con il codice di LLaVA-NeXT (come il baseline).
# Porta con se' tokenizers 0.15.2 e huggingface_hub < 1.0 (wheel anche per Python 3.13).
# accelerate serve per device_map="auto". peft non serve in inferenza e non si installa.
# torch resta quello di Colab, come nel baseline.
# Gli avvisi rossi di pip su gradio, sentence-transformers o peft sono attesi:
# sono pacchetti di Colab che questo notebook non importa.
%pip install -q "transformers==4.37.2" "accelerate>=0.29.1,<2" av timm einops sentencepiece

# Repo ufficiale LLaVA-NeXT, fissato a un commit: se il repo cambia, il codice resta quello del baseline
LLAVA_DIR = "/content/LLaVA-NeXT"
LLAVA_COMMIT = "bce12e479bc4dfee2b9c50c88137b01ff51bd483"   # ultimo commit del repo (giugno 2026)
%cd /content
if not os.path.exists(LLAVA_DIR):
    !git clone -q https://github.com/LLaVA-VL/LLaVA-NeXT.git {LLAVA_DIR}
!git -C {LLAVA_DIR} checkout -q {LLAVA_COMMIT}
!git -C {LLAVA_DIR} log -1 --format="LLaVA-NeXT: commit %h del %cd"
%cd /content/LLaVA-NeXT

# installo il pacchetto llava senza dipendenze (non deve toccare transformers)
%pip install -q -e ".[train]" --no-deps
if LLAVA_DIR not in sys.path:
    sys.path.insert(0, LLAVA_DIR)

from importlib.metadata import version
_vers = {k: version(k) for k in ["torch", "transformers", "tokenizers", "accelerate", "huggingface_hub"]}
print(_vers)
assert _vers["transformers"] == "4.37.2", f"transformers {_vers['transformers']}: controllare l'output di pip sopra"

In [ ]:
import sys, torch, transformers
# Se nella sessione era gia' stato importato un altro transformers (es. il notebook VideoLLaMA 2),
# in memoria c'e' ancora quello: serve una sessione nuova.
assert transformers.__version__ == "4.37.2", (
    f"transformers {transformers.__version__} in memoria: Runtime > Disconnetti ed elimina runtime, poi da capo")
assert torch.cuda.is_available(), "CUDA non disponibile: il runtime non ha la GPU"

if "/content/LLaVA-NeXT" not in sys.path:
    sys.path.insert(0, "/content/LLaVA-NeXT")

# Loader ufficiale lmms-lab: carica pesi, tokenizer e image_processor
# con le chiavi corrette per LLaVA-Video-7B-Qwen2
from llava.model.builder import load_pretrained_model

model_path = "lmms-lab/LLaVA-Video-7B-Qwen2"
model_name = "llava_qwen"   # nome architettura richiesto dal builder

# carico il modello (circa 16 GB da scaricare, piu' la vision tower SigLIP)
tokenizer, model, image_processor, max_length = load_pretrained_model(
    model_path,
    None,           # model_base: None per modelli stand-alone
    model_name,
    torch_dtype="bfloat16",
    device_map="auto",
    attn_implementation="sdpa",
)

# mettiamo in modalita' evaluation
model.eval()

# Tutto il modello deve stare sulla GPU: con device_map="auto", se la memoria non basta,
# accelerate sposta dei layer sulla CPU senza errori ma rende l'inferenza lentissima.
fuori = {str(p.device) for p in model.parameters() if p.device.type != "cuda"}
assert not fuori, f"parte del modello e' su {fuori} (offload): serve piu' VRAM libera"
device = next(model.parameters()).device

# L4: logits solo per l'ultima posizione.
# transformers 4.37 nel prefill calcola i logits su TUTTA la sequenza (circa 6.900 token con
# 32 fotogrammi) e li converte in float32: 6.900 x 152.064 voci = circa 5,9 GiB in piu' oltre
# ai 15 GiB di pesi. Su A100 c'era spazio, su L4 (22 GiB utili) e' out of memory.
# Con decoding greedy si usa solo la riga dell'ultima posizione, quindi l'hook passa al lm_head
# solo quella. Gli hidden state sono gli stessi; puo' cambiare solo l'arrotondamento del
# prodotto finale, che conta solo se due lettere sono praticamente alla pari.
if globals().get("_hook_lm_head") is not None:
    _hook_lm_head.remove()
_hook_lm_head = model.lm_head.register_forward_pre_hook(lambda modulo, args: (args[0][:, -1:, :],))

print(f"LLaVA-Video-7B-Qwen2 caricato su {device}")
print(f"VRAM occupata: {torch.cuda.memory_allocated() / 1024**3:.1f} GiB "
      f"su {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")

---
# Parte 1 — Riproduzione della baseline HD-EPIC (LLaVA-Video 7B)

Stesso identico protocollo del notebook `01_videollama2_inference.ipynb`, applicato al
secondo modello. Il valore da verificare e' quello del paper HD-EPIC  per LLaVA-Video 7B: **38.7%**, contro un random baseline del 20%.


L'unica parte specifica di questo notebook e' la funzione `llava_infer`.

## 1.1 Motore di valutazione

La cella qui sotto importa il modulo `src/hdepic/eval.py` (inferenza e report), copiato su Drive
da `scripts/sync_drive.sh`.

In [ ]:
# Il motore di valutazione sta in src/hdepic/eval.py.
# Su Drive ce lo porta scripts/sync_drive.sh, lanciato dal Mac: prima di questa cella.
import os, sys, importlib
BASE = "/content/drive/MyDrive/Progetto Deep Learning 1"
SRC = os.path.join(BASE, "src")
assert os.path.exists(os.path.join(SRC, "hdepic", "eval.py")), (
    f"Manca {SRC}/hdepic/eval.py su Drive: lanciare scripts/sync_drive.sh dal Mac")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
from hdepic import eval as H
importlib.reload(H)   # se la cella viene rieseguita dopo una modifica del modulo


## 1.2 Percorsi e indicizzazione dei file

Indicizzo i file

In [ ]:
import os, glob

BASE = "/content/drive/MyDrive/Progetto Deep Learning 1"
ANN_DIR   = os.path.join(BASE, "data", "hd-epic")
OUT_DIR   = os.path.join(BASE, "results", "baseline")
FRAME_DIR = os.path.join(BASE, "frames_image_nutrition_estimation")
CLIP_DIR  = os.path.join(BASE, "frames_nutrition_change")   # clip di nutrition_change (extract_clip_frames.py)

VIDEO_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" not in os.path.basename(d).lower()
                     and os.path.basename(d).lower().startswith(("video", "videos")))
PANEL_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" in os.path.basename(d).lower())

print("cartelle video :", [os.path.basename(d) for d in VIDEO_ROOTS])
print("cartelle panels:", [os.path.basename(d) for d in PANEL_ROOTS])

VIDEO_INDEX = H.build_video_index(VIDEO_ROOTS)
PANEL_INDEX = H.build_panels_index(PANEL_ROOTS)
print(f"\nindicizzati: {len(VIDEO_INDEX['sampled'])} video campionati, "
      f"{len(VIDEO_INDEX['full'])} video integrali, {len(PANEL_INDEX)} cartelle di pannelli")
n_clip = len(glob.glob(os.path.join(CLIP_DIR, "*", "frame_000.jpg")))
print(f"clip di nutrition_change in {os.path.basename(CLIP_DIR)}: {n_clip} (attese 50)")

READABLE = {}
cov, dettagli = H.preflight(ANN_DIR, VIDEO_INDEX, cache=READABLE, frame_cache=FRAME_DIR,
                              clip_cache=CLIP_DIR)
print()
print(cov.to_string(index=False))

## 1.3 Il modello

LLaVA-Video vuole i fotogrammi gia' estratti, quindi la funzione di inferenza deve gestire
due casi: il path di un mp4 (da cui campionare i frame con PyAV) e una lista di path di
immagini (i 5 fotogrammi del task `image_nutrition_estimation`, i 128 fotogrammi di una clip
di `nutrition_change`, oppure i pannelli della Parte 2). Le liste piu' lunghe di `N_FRAMES`
vengono sottocampionate uniformemente.

**Budget di fotogrammi.** `N_FRAMES = 32` vale per entrambi i casi. Il numero non e'
arbitrario: i pannelli generati da `mosaic.py` sono 32 per video, quindi fissando 32 anche
per il video liscio il confronto della Parte 2 avviene a **parita' di finestra di contesto**,
che e' esattamente la condizione in cui il paper dei Video Panels rivendica il guadagno.
Con 32 frame lisci il modello copre 32 fotogrammi; con 32 pannelli 2x2 ne copre 128.

In [ ]:
import av, torch
import numpy as np
from PIL import Image

from llava.conversation import conv_templates
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llava.mm_utils import tokenizer_image_token

N_FRAMES = 32
MAX_NEW_TOKENS = 32
USE_SYSTEM_PROMPT = None     # None = prompt minimale, fedele al paper

# estrae i fotogrammi in quanto LlavaVideo vuole i fotogrammi già estratti
def sample_video_frames(video_path, n=N_FRAMES):
    """Estrae n fotogrammi uniformi con PyAV (il codebase ufficiale di LLaVA-Video usa PyAV)."""
    container = av.open(video_path)
    stream = container.streams.video[0]
    total = stream.frames
    if total == 0:                                  # metadato assente: li conto
        total = sum(1 for _ in container.decode(video=0))
        container.close()
        container = av.open(video_path)
    voluti = set(np.linspace(0, total - 1, min(n, total), dtype=int).tolist())
    frames = []
    for i, frame in enumerate(container.decode(video=0)):
        if i in voluti:
            frames.append(frame.to_ndarray(format="rgb24"))
        if len(frames) == len(voluti):
            break
    container.close()
    return frames


def _immagini(visual, n=N_FRAMES):
    """Normalizza l'input visivo in una lista di PIL.Image."""
    if isinstance(visual, list):                    # frame estratti o pannelli
        paths = visual
        if len(paths) > n:                          # sottocampiono a parita' di budget
            sel = np.linspace(0, len(paths) - 1, n, dtype=int)
            paths = [paths[i] for i in sel]
        return [Image.open(p).convert("RGB") for p in paths]
    return [Image.fromarray(f) for f in sample_video_frames(visual, n)]


# inferenza per llava video
def llava_infer(visual, prompt):
    """visual: path di un mp4, oppure lista di path di immagini."""
    immagini = _immagini(visual)
    video_tensor = image_processor.preprocess(immagini, return_tensors="pt")["pixel_values"]
    video_tensor = video_tensor.to(device=device, dtype=torch.bfloat16)

    # il token immagine vale anche per i video nel codebase ufficiale
    conv = conv_templates["qwen_1_5"].copy()
    conv.append_message(conv.roles[0], f"{DEFAULT_IMAGE_TOKEN}\n{prompt}")
    conv.append_message(conv.roles[1], None)
    input_ids = tokenizer_image_token(conv.get_prompt(), tokenizer,
                                      IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0).to(device)

    with torch.no_grad():
        out = model.generate(input_ids, images=[video_tensor], image_sizes=None,
                             modalities=["video"],
                             do_sample=False, temperature=None,   # greedy: deterministico
                             max_new_tokens=MAX_NEW_TOKENS, use_cache=True)

    testo = tokenizer.decode(out[0], skip_special_tokens=False)
    marker = "<|im_start|>assistant\n"
    if marker in testo:
        return testo.split(marker)[-1].replace("<|im_end|>", "").strip()
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()


print(f"pronto - {N_FRAMES} fotogrammi per domanda, decoding greedy")

pronto - 32 fotogrammi per domanda, decoding greedy


## 1.5 Inferenza completa

In [ ]:
baseline = H.run_all(list(H.TASK_FILES), ANN_DIR, VIDEO_INDEX, llava_infer,
                     out_dir=OUT_DIR, tag="llavavideo_baseline_clip",   # nuovo tag: nutrition_change ora usa la clip
                     system=USE_SYSTEM_PROMPT,
                     frame_extractor=H.extract_frame, frame_cache=FRAME_DIR,
                     readable_cache=READABLE, clip_cache=CLIP_DIR)#inferenza sui video (frame estratti con le funzioni precedenti)

print(f"\nCompletate {len(baseline)} domande")

In [11]:
rep_base = H.accuracy_report(baseline,
                             label="LLaVA-Video 7B - baseline (video)",
                             paper_ref="LLaVA-Video 7B",
                             ann_dir=ANN_DIR)

  LLaVA-Video 7B - baseline (video)   (n = 200 domande valutate)
                            corrette  valutate  accuracy_%  su_totale
task                                                                 
image_nutrition_estimation        26       100        26.0        100
nutrition_change                  11        50        22.0         50
video_nutrition_estimation        34        50        68.0         50

  Nutrition, metrica del paper (media per prototipo) : 38.7%
  Nutrition, accuracy aggregata sulle domande        : 35.5%
  Baseline random                                    : 20.0%
  Paper, LLaVA-Video 7B              : 38.7%   (delta +0.0)

  Risposte non interpretate o in errore: 0/200 (0.0%)
  Come e' stata letta la risposta:
    {'leading_letter': 200}

  Accuracy per tipo di domanda:
                  corrette  valutate  accuracy_%
q_type                                          
calories                16        41        39.0
carbs                   18        35       

---
# Parte 2 — Video Panels

Esperimento basato sul Paper dei Panels: 32 pannelli 2x2 al posto di 32 fotogrammi lisci, cioe'
128 fotogrammi coperti invece di 32 a parita' di finestra di contesto.

Anche qui solo `video_nutrition_estimation` (`H.PANEL_TASKS`): `image_nutrition_estimation` ha
come input 5 fotogrammi isolati presi da video diversi, e `nutrition_change` una clip di 1-27 s,
sotto la soglia di 128 s oltre la quale il paper applica i pannelli.

In [ ]:
cov_p, dett_p = H.preflight(ANN_DIR, VIDEO_INDEX, tasks=H.PANEL_TASKS,
                            panels_index=PANEL_INDEX, input_mode="panels",
                            cache=READABLE)
print(cov_p.to_string(index=False))

panels_run = H.run_all(H.PANEL_TASKS, ANN_DIR, VIDEO_INDEX, llava_infer,
                       out_dir=OUT_DIR, tag="llavavideo_panels_v2",   # nuovo tag: run pulito sui pannelli attuali (P06 rigenerato)
                       panels_index=PANEL_INDEX, input_mode="panels",
                       system=USE_SYSTEM_PROMPT, readable_cache=READABLE)

rep_pan = H.accuracy_report(panels_run,
                            label="LLaVA-Video 7B - Video Panels 2x2",
                            ann_dir=ANN_DIR)

In [ ]:
comuni = set(panels_run["key"]) & set(baseline["key"])
b = baseline[baseline["key"].isin(comuni)]
p = panels_run[panels_run["key"].isin(comuni)]
print(f"domande confrontate a parita' di input: {len(comuni)}\n")

tab = H.compare_runs({"baseline (video)": b, "panels 2x2": p},
                     "LLaVA-Video 7B: video liscio vs pannelli 2x2")

## 2.1 Salvataggio dei risultati

I CSV finiscono nella stessa cartella di quelli di VideoLLaMA 2, con il nome del modello nel
prefisso: da li' si costruisce la tabella finale con le due righe da confrontare con il paper.

In [14]:
import pandas as pd

tutto = pd.concat([baseline.assign(run="baseline"),
                   panels_run.assign(run="panels")], ignore_index=True)
csv = os.path.join(OUT_DIR, "llavavideo_risultati_completi.csv")
tutto.to_csv(csv, index=False)

riepilogo = pd.DataFrame({
    "run": ["baseline (video)", "panels 2x2"],
    "nutrition_media_prototipi_%": [rep_base["nutrition_paper"], rep_pan["nutrition_paper"]],
    "nutrition_aggregata_%":       [rep_base["nutrition_pooled"], rep_pan["nutrition_pooled"]],
    "domande_valutate":            [len(baseline), len(panels_run)],
})
riepilogo.to_csv(os.path.join(OUT_DIR, "llavavideo_riepilogo.csv"), index=False)

print(riepilogo.to_string(index=False))
print(f"\nsalvato: {csv}")
print(f"paper, LLaVA-Video 7B su Nutrition: {H.PAPER_NUTRITION['LLaVA-Video 7B']}%  | random: 20.0%")

             run  nutrition_media_prototipi_%  nutrition_aggregata_%  domande_valutate
baseline (video)                         38.7                   35.5               200
      panels 2x2                         43.2                   43.4                99

salvato: /content/drive/MyDrive/Progetto Deep Learning 1/risultati/llavavideo_risultati_completi.csv
paper, LLaVA-Video 7B su Nutrition: 38.7%  | random: 20.0%
